# OPFData — Dataset Overview (.tar.gz)

This notebook inspects the **OPFData** [dataset](https://huggingface.co/datasets/AI-grids/OPFData/tree/main). From the Hugging Face's README:

GridOpt is a large dataset of solved Optimal Power Flow (OPF) problems.
The source of the original problems is the [pglib-opf](https://github.com/power-grid-lib/pglib-opf/) dataset.

Example problems are generated by perturbing the load and the topology of the original problem.
They are solved using the [PowerModels.jl](https://github.com/lanl-ansi/PowerModels.jl) package and [IPOPT](https://coin-or.github.io/Ipopt/) solver.
See the associated [technical report](https://arxiv.org/abs/2406.07234) for details on the dataset generation.

Each grid contains 300k solved examples with only load perturbations, and 300k solved examples with topological perturbations, where a line/transformer/generator is removed randomly.

Users of GridOpt are encouraged to cite the original source documents indicated in the [pglib-opf file headers](https://github.com/power-grid-lib/pglib-opf/) and [arxiv report](https://arxiv.org/abs/1908.02788).

## Notes
- The datasets reside in `./home/user/data/datasets/OPFData/`.
- We've uploaded four IEEE cases: 14, 30, 57 and 118, each with their respective sub-directory (e.g. `../pglib_opf_case14_ieee/`).
- Each case contains multiple `.tar.gz` files (variation of `.zip`), named as `group_0.tar.gz`, `group_1.tar.gz`, etc.
- Each `.tar.gz` file contains multiple `.json` files, names as `merged_1.json`, `merged_2.json`, etc.
- This notebook serves as an EDA for one of those `.json` files.

---
#### Bootstrap

In [1]:
from _notebook_bootstrap import bootstrap

repo_root, datasets_root = bootstrap()
repo_root, datasets_root

(PosixPath('/home/user/workspace/datasets-utilities'),
 PosixPath('/home/user/data/datasets'))

---
#### Imports

In [2]:
import json
import tarfile
from pathlib import Path
import pandas as pd

---
#### Load dataset object

In [3]:
# Select your file (can be changed to any other snapshot path under datasets_root)
relpath = "OPFData/pglib_opf_case14_ieee/group_0.tar.gz"

datasets_root = Path(datasets_root)
archive_path = datasets_root / relpath

assert archive_path.exists(), f"Archive not found: {archive_path}"

print(f"Archive path: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 ** 2):,.2f} MB")

Archive path: /home/user/data/datasets/OPFData/pglib_opf_case14_ieee/group_0.tar.gz
Archive size: 19.57 MB


---
#### Inspect archive contents

In [4]:
with tarfile.open(archive_path, "r:gz") as tar:
    members = [m for m in tar.getmembers() if m.isfile()]

archive_index = pd.DataFrame(
    {
        "name": [m.name for m in members],
        "size_mb": [m.size / (1024 ** 2) for m in members],
    }
)

json_members = archive_index.loc[
    archive_index["name"].str.endswith(".json"), "name"
].tolist()

display(archive_index.head())

print(f"Files in archive: {len(archive_index)}")
print(f"JSON files in archive: {len(json_members)}")

,name,size_mb
0,group_0/merged_19.json,0.900735
1,group_0/merged_17.json,0.899572
2,group_0/merged_48.json,0.900809
3,group_0/merged_97.json,0.899900
4,group_0/merged_3.json,0.899799


Files in archive: 100
JSON files in archive: 100


---
#### Select JSON snapshot

In [5]:
# Select one JSON file inside the archive
target = json_members[0]

print(f"Selected JSON file: {target}")

Selected JSON file: group_0/merged_19.json


---
#### Load JSON Snapshot

In [6]:
with tarfile.open(archive_path, "r:gz") as tar:
    with tar.extractfile(target) as f:
        data = json.load(f)

print(type(data))
print(f"Number of OPF examples in this JSON file: {len(data)}")

first_id = next(iter(data))
first_example = data[first_id]

print(f"First example ID: {first_id}")

<class 'dict'>
Number of OPF examples in this JSON file: 150
First example ID: 8440


---
#### JSON structure helper

In [7]:
def print_json_structure(obj, indent=0, max_depth=6):
    prefix = "  " * indent

    if indent > max_depth:
        print(prefix + "...")
        return

    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict):
                print(f"{prefix}{key}: dict")
                print_json_structure(value, indent + 1, max_depth)

            elif isinstance(value, list):
                print(f"{prefix}{key}: list[{len(value)}]")
                if len(value) > 0:
                    print_json_structure(value[0], indent + 1, max_depth)

            else:
                print(f"{prefix}{key}: {type(value).__name__}")

    elif isinstance(obj, list):
        print(f"{prefix}list[{len(obj)}]")
        if len(obj) > 0:
            print_json_structure(obj[0], indent + 1, max_depth)

    else:
        print(f"{prefix}{type(obj).__name__}")

---
#### Inspect 1st example structure

In [8]:
print_json_structure({first_id: first_example}, max_depth=6)

8440: dict
  grid: dict
    nodes: dict
      bus: list[14]
        list[4]
          float
      generator: list[5]
        list[11]
          float
      load: list[11]
        list[2]
          float
      shunt: list[1]
        list[2]
          float
    edges: dict
      ac_line: dict
        senders: list[16]
          int
        receivers: list[16]
          int
        features: list[16]
          list[9]
            float
      transformer: dict
        senders: list[3]
          int
        receivers: list[3]
          int
        features: list[3]
          list[11]
            float
      generator_link: dict
        senders: list[5]
          int
        receivers: list[5]
          int
      load_link: dict
        senders: list[11]
          int
        receivers: list[11]
          int
      shunt_link: dict
        senders: list[1]
          int
        receivers: list[1]
          int
    context: list[1]
      list[1]
        list[1]
          float
  solution: dic

#### Feature meaning

Each example contains three main objects: `grid`, `solution`, and `metadata`.

- `grid` contains the OPF input instance.
- `solution` contains the solved OPF output.
- `metadata` contains scalar metadata about the solved instance.

For node and edge feature matrices:
- Rows correspond to entities, such as buses, generators, loads, lines, or transformers.
- Columns correspond to features.
- For edge objects, `senders` and `receivers` define the graph connectivity.

The following cells define the column names used by OPFData and print small examples from the selected JSON object.

#### Tree-like structure of an example

```bash
8440
├── grid
│   ├── nodes
│   │   ├── bus
│   │   ├── generator
│   │   ├── load
│   │   └── shunt
│   ├── edges
│   │   ├── ac_line
│   │   │   ├── senders
│   │   │   ├── receivers
│   │   │   └── features
│   │   ├── transformer
│   │   │   ├── senders
│   │   │   ├── receivers
│   │   │   └── features
│   │   ├── generator_link
│   │   │   ├── senders
│   │   │   └── receivers
│   │   ├── load_link
│   │   │   ├── senders
│   │   │   └── receivers
│   │   └── shunt_link
│   │       ├── senders
│   │       └── receivers
│   └── context
├── solution
│   ├── nodes
│   │   ├── bus
│   │   └── generator
│   └── edges
│       ├── ac_line
│       │   ├── senders
│       │   ├── receivers
│       │   └── features
│       └── transformer
│           ├── senders
│           ├── receivers
│           └── features
└── metadata
    └── objective
```

---
#### Define OPFData feature names

In [9]:
FEATURE_NAMES = {
    ("grid", "nodes", "bus"): [
        "base_kv",
        "bus_type",
        "vmin",
        "vmax",
    ],

    ("grid", "nodes", "generator"): [
        "mbase",
        "pg",
        "pmin",
        "pmax",
        "qg",
        "qmin",
        "qmax",
        "vg",
        "cost_squared",
        "cost_linear",
        "cost_offset",
    ],

    ("grid", "nodes", "load"): [
        "pd",
        "qd",
    ],

    ("grid", "nodes", "shunt"): [
        "bs",
        "gs",
    ],

    ("grid", "edges", "ac_line"): [
        "angmin",
        "angmax",
        "b_fr",
        "b_to",
        "br_r",
        "br_x",
        "rate_a",
        "rate_b",
        "rate_c",
    ],

    ("grid", "edges", "transformer"): [
        "angmin",
        "angmax",
        "br_r",
        "br_x",
        "rate_a",
        "rate_b",
        "rate_c",
        "tap",
        "shift",
        "b_fr",
        "b_to",
    ],

    ("solution", "nodes", "bus"): [
        "va",
        "vm",
    ],

    ("solution", "nodes", "generator"): [
        "pg",
        "qg",
    ],

    ("solution", "edges", "ac_line"): [
        "pt",
        "qt",
        "pf",
        "qf",
    ],

    ("solution", "edges", "transformer"): [
        "pt",
        "qt",
        "pf",
        "qf",
    ],
}

---
#### Define short feature descriptions

In [10]:
FEATURE_DESCRIPTIONS = {
    "base_kv": "Base voltage in kV",
    "bus_type": "Bus type: PQ=1, PV=2, reference=3, inactive=4",
    "vmin": "Minimum voltage magnitude",
    "vmax": "Maximum voltage magnitude",

    "mbase": "Generator total MVA base",
    "pg": "Real power generation",
    "pmin": "Minimum real power generation",
    "pmax": "Maximum real power generation",
    "qg": "Reactive power generation",
    "qmin": "Minimum reactive power generation",
    "qmax": "Maximum reactive power generation",
    "vg": "Initial generator voltage magnitude",
    "cost_squared": "Quadratic generation cost coefficient",
    "cost_linear": "Linear generation cost coefficient",
    "cost_offset": "Constant generation cost coefficient",

    "pd": "Real power demand",
    "qd": "Reactive power demand",

    "bs": "Shunt susceptance",
    "gs": "Shunt conductance",

    "angmin": "Minimum angle difference",
    "angmax": "Maximum angle difference",
    "b_fr": "Line charging susceptance at from bus",
    "b_to": "Line charging susceptance at to bus",
    "br_r": "Branch series resistance",
    "br_x": "Branch series reactance",
    "rate_a": "Long-term thermal rating",
    "rate_b": "Short-term thermal rating",
    "rate_c": "Emergency thermal rating",
    "tap": "Transformer off-nominal turns ratio",
    "shift": "Transformer phase shift angle",

    "va": "Voltage angle",
    "vm": "Voltage magnitude",

    "pt": "Active power withdrawn at to bus",
    "qt": "Reactive power withdrawn at to bus",
    "pf": "Active power withdrawn at from bus",
    "qf": "Reactive power withdrawn at from bus",

    "baseMVA": "System-wide MVA base",
    "objective": "AC-OPF objective value",
}

---
#### Build feature dictionary table

In [11]:
feature_dictionary_rows = []

for key, feature_names in FEATURE_NAMES.items():
    section, object_type, component = key

    for position, feature_name in enumerate(feature_names):
        feature_dictionary_rows.append(
            {
                "section": section,
                "object_type": object_type,
                "component": component,
                "position": position,
                "feature": feature_name,
                "description": FEATURE_DESCRIPTIONS.get(feature_name, ""),
            }
        )

feature_dictionary = pd.DataFrame(feature_dictionary_rows)

display(feature_dictionary)

,section,object_type,component,position,feature,description
0,grid,nodes,bus,0,base_kv,Base voltage in kV
1,grid,nodes,bus,1,bus_type,"Bus type: PQ=1, PV=2, reference=3, inactive=4"
2,grid,nodes,bus,2,vmin,Minimum voltage magnitude
3,grid,nodes,bus,3,vmax,Maximum voltage magnitude
4,grid,nodes,generator,0,mbase,Generator total MVA base
5,grid,nodes,generator,1,pg,Real power generation
6,grid,nodes,generator,2,pmin,Minimum real power generation
7,grid,nodes,generator,3,pmax,Maximum real power generation
8,grid,nodes,generator,4,qg,Reactive power generation
9,grid,nodes,generator,5,qmin,Minimum reactive power generation


#### Observed structure in the selected example

The next cell checks the observed number of rows and columns for each feature matrix in the selected example.

For topological-perturbation data, the number of AC lines, transformers, or generators may vary across examples because components can be removed.

---
#### Check observed shapes against expected feature names

In [12]:
def get_feature_matrix(example, section, object_type, component):
    obj = example[section][object_type][component]

    if object_type == "edges":
        return obj["features"]

    return obj


shape_rows = []

for key, feature_names in FEATURE_NAMES.items():
    section, object_type, component = key
    matrix = get_feature_matrix(first_example, section, object_type, component)

    n_rows = len(matrix)
    n_observed_features = len(matrix[0]) if n_rows > 0 else 0
    n_expected_features = len(feature_names)

    shape_rows.append(
        {
            "section": section,
            "object_type": object_type,
            "component": component,
            "n_rows": n_rows,
            "observed_features": n_observed_features,
            "expected_features": n_expected_features,
            "matches_expected": n_observed_features == n_expected_features,
        }
    )

shape_summary = pd.DataFrame(shape_rows)

display(shape_summary)

,section,object_type,component,n_rows,observed_features,expected_features,matches_expected
0,grid,nodes,bus,14,4,4,True
1,grid,nodes,generator,5,11,11,True
2,grid,nodes,load,11,2,2,True
3,grid,nodes,shunt,1,2,2,True
4,grid,edges,ac_line,16,9,9,True
5,grid,edges,transformer,3,11,11,True
6,solution,nodes,bus,14,2,2,True
7,solution,nodes,generator,5,2,2,True
8,solution,edges,ac_line,16,4,4,True
9,solution,edges,transformer,3,4,4,True


---
#### Helper for printing example feature tables

In [13]:
def feature_matrix_to_df(example, section, object_type, component):
    matrix = get_feature_matrix(example, section, object_type, component)
    columns = FEATURE_NAMES[(section, object_type, component)]

    return pd.DataFrame(matrix, columns=columns)


def edge_feature_matrix_to_df(example, section, component):
    edge_data = example[section]["edges"][component]

    df = pd.DataFrame(
        edge_data["features"],
        columns=FEATURE_NAMES[(section, "edges", component)],
    )

    df.insert(0, "receiver", edge_data["receivers"])
    df.insert(0, "sender", edge_data["senders"])

    return df

#### `grid.nodes` examples

The `grid.nodes` object contains the input features for buses, generators, loads, and shunts.

In [14]:
for component in ["bus", "generator", "load", "shunt"]:
    print(f"grid.nodes.{component}")
    display(feature_matrix_to_df(first_example, "grid", "nodes", component).head())

grid.nodes.bus


,base_kv,bus_type,vmin,vmax
0,1.0,3.0,0.94,1.06
1,1.0,2.0,0.94,1.06
2,1.0,2.0,0.94,1.06
3,1.0,1.0,0.94,1.06
4,1.0,1.0,0.94,1.06


grid.nodes.generator


,mbase,pg,pmin,pmax,qg,qmin,qmax,vg,cost_squared,cost_linear,cost_offset
0,100.0,1.700,0.0,3.40,0.05,0.00,0.10,1.0,0.0,792.0951,0.0
1,100.0,0.295,0.0,0.59,0.00,-0.30,0.30,1.0,0.0,2326.9494,0.0
2,100.0,0.000,0.0,0.00,0.20,0.00,0.40,1.0,0.0,0.0000,0.0
3,100.0,0.000,0.0,0.00,0.09,-0.06,0.24,1.0,0.0,0.0000,0.0
4,100.0,0.000,0.0,0.00,0.09,-0.06,0.24,1.0,0.0,0.0000,0.0


grid.nodes.load


,pd,qd
0,0.258666,0.134248
1,0.939603,0.207592
2,0.424162,-0.031379
3,0.063583,0.015232
4,0.113389,0.082285


grid.nodes.shunt


,bs,gs
0,0.19,0.0


#### `grid.edges` examples

The `grid.edges` object contains the input features for AC lines and transformers.

For `ac_line` and `transformer`, each row is an edge. The `sender` and `receiver` columns identify the connected buses.

In [15]:
for component in ["ac_line", "transformer"]:
    print(f"grid.edges.{component}")
    display(edge_feature_matrix_to_df(first_example, "grid", component).head())

grid.edges.ac_line


,sender,receiver,angmin,angmax,b_fr,b_to,br_r,br_x,rate_a,rate_b,rate_c
0,0,1,-0.523599,0.523599,0.0264,0.0264,0.01938,0.05917,4.72,4.72,4.72
1,0,4,-0.523599,0.523599,0.0246,0.0246,0.05403,0.22304,1.28,1.28,1.28
2,1,2,-0.523599,0.523599,0.0219,0.0219,0.04699,0.19797,1.45,1.45,1.45
3,1,3,-0.523599,0.523599,0.0170,0.0170,0.05811,0.17632,1.58,1.58,1.58
4,1,4,-0.523599,0.523599,0.0173,0.0173,0.05695,0.17388,1.61,1.61,1.61


grid.edges.transformer


,sender,receiver,angmin,angmax,br_r,br_x,rate_a,rate_b,rate_c,tap,shift,b_fr,b_to
0,3,6,-0.523599,0.523599,0.0,0.20912,1.41,1.41,1.41,0.978,0.0,0.0,0.0
1,3,8,-0.523599,0.523599,0.0,0.55618,0.53,0.53,0.53,0.969,0.0,0.0,0.0
2,4,5,-0.523599,0.523599,0.0,0.25202,1.17,1.17,1.17,0.932,0.0,0.0,0.0


#### Artificial link edges

The `generator_link`, `load_link`, and `shunt_link` objects connect non-bus nodes to bus nodes.

These links do not have feature matrices. They only contain `senders` and `receivers`.

In [16]:
link_rows = []

for component in ["generator_link", "load_link", "shunt_link"]:
    edge_data = first_example["grid"]["edges"][component]

    for sender, receiver in zip(edge_data["senders"], edge_data["receivers"]):
        link_rows.append(
            {
                "link_type": component,
                "sender": sender,
                "receiver": receiver,
            }
        )

link_edges = pd.DataFrame(link_rows)

display(link_edges)

,link_type,sender,receiver
0,generator_link,0,0
1,generator_link,1,1
2,generator_link,2,2
3,generator_link,3,5
4,generator_link,4,7
5,load_link,0,1
6,load_link,1,2
7,load_link,2,3
8,load_link,3,4
9,load_link,4,5


#### `solution` examples

The `solution` object contains the OPF solution.

- `solution.nodes.bus` gives solved bus voltage angle and magnitude.
- `solution.nodes.generator` gives solved generator real and reactive power.
- `solution.edges.ac_line` and `solution.edges.transformer` give solved branch flows.

In [17]:
for component in ["bus", "generator"]:
    print(f"solution.nodes.{component}")
    display(feature_matrix_to_df(first_example, "solution", "nodes", component).head())

solution.nodes.bus


,va,vm
0,-1.862032e-34,1.060000
1,-1.165005e-01,1.027886
2,-2.858595e-01,1.004760
3,-2.649994e-01,1.000617
4,-1.257835e-01,1.009962


solution.nodes.generator


,pg,qg
0,2.771991e+00,0.064534
1,-9.985634e-09,0.300000
2,0.000000e+00,0.400000
3,0.000000e+00,0.149825
4,0.000000e+00,0.137954


In [18]:
for component in ["ac_line", "transformer"]:
    print(f"solution.edges.{component}")
    display(edge_feature_matrix_to_df(first_example, "solution", component).head())

solution.edges.ac_line


,sender,receiver,pt,qt,pf,qf
0,0,1,-2.061127,0.214126,2.140126,-0.030487
1,0,4,-0.611943,-0.065513,0.631866,0.095020
2,1,2,-0.842006,0.135015,0.876155,-0.036392
3,1,3,-0.805413,0.157871,0.844837,-0.073231
4,1,4,-0.080770,-0.095041,0.081469,0.061250


solution.edges.transformer


,sender,receiver,pt,qt,pf,qf
0,3,6,-0.179652,0.072357,0.179652,-0.065064
1,3,8,-0.103099,-0.000963,0.103099,0.006529
2,4,5,-0.629131,-0.055844,0.629131,0.145322


#### `metadata` example

The `metadata` object contains scalar information about the solved OPF instance.

In [19]:
metadata_df = pd.DataFrame(
    [
        {
            "field": "objective",
            "value": first_example["metadata"]["objective"],
            "description": FEATURE_DESCRIPTIONS["objective"],
        }
    ]
)

display(metadata_df)

,field,value,description
0,objective,2195.680833,AC-OPF objective value


In [20]:
context_df = pd.DataFrame(
    [
        {
            "field": "baseMVA",
            "value": first_example["grid"]["context"][0][0][0],
            "description": FEATURE_DESCRIPTIONS["baseMVA"],
        }
    ]
)

display(context_df)

,field,value,description
0,baseMVA,100.0,System-wide MVA base
